<a href="https://colab.research.google.com/github/AoWangPhilly/red-handed/blob/master/get_yearly.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NYPD Complaint Data Historic

In [ ]:
%pip install pyexcelerate

In [ ]:
import pandas as pd
import numpy as np
import json
from pyexcelerate import Workbook
from google.colab import drive

In [ ]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%cd '/content/drive/My Drive/Colab Notebooks/red-handed/src/data'

/content/drive/My Drive/Colab Notebooks/red-handed/src/data


In [ ]:
schema_pwd = 'nyc_crime_schema.json'

with open(schema_pwd) as f:
    schema_dict = json.load(f)
schema = pd.Series(data=schema_dict)

In [ ]:
schema.index = schema.index.str.upper()

In [ ]:
schema

CMPLNT_NUM           Randomly generated persistent ID for each comp...
CMPLNT_FR_DT         Exact date of occurrence for the reported even...
CMPLNT_FR_TM         Exact time of occurrence for the reported even...
CMPLNT_TO_DT         Ending date of occurrence for the reported eve...
CMPLNT_TO_TM         Ending time of occurrence for the reported eve...
ADDR_PCT_CD                The precinct in which the incident occurred
RPT_DT                               Date event was reported to police
KY_CD                          Three digit offense classification code
OFNS_DESC            Description of offense corresponding with key ...
PD_CD                Three digit internal classification code (more...
PD_DESC              Description of internal classification corresp...
CRM_ATPT_CPTD_CD     Indicator of whether crime was successfully co...
LAW_CAT_CD            Level of offense: felony, misdemeanor, violation
BORO_NM              The name of the borough in which the incident ...
LOC_OF

In [ ]:
crime_date_pwd = 'NYPD_Complaint_Data_Historic.csv'
crime_df_yrs = pd.read_csv(crime_date_pwd)
crime_df_yrs.set_index('CMPLNT_NUM', inplace=True)

/usr/local/lib/python3.6/dist-packages/IPython/core/interactiveshell.py:2718: DtypeWarning: Columns (18,20) have mixed types.Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


In [ ]:
crime_df_yrs.head()

,CMPLNT_FR_DT,CMPLNT_FR_TM,CMPLNT_TO_DT,CMPLNT_TO_TM,ADDR_PCT_CD,RPT_DT,KY_CD,OFNS_DESC,PD_CD,PD_DESC,CRM_ATPT_CPTD_CD,LAW_CAT_CD,BORO_NM,LOC_OF_OCCUR_DESC,PREM_TYP_DESC,JURIS_DESC,JURISDICTION_CODE,PARKS_NM,HADEVELOPT,HOUSING_PSA,X_COORD_CD,Y_COORD_CD,SUSP_AGE_GROUP,SUSP_RACE,SUSP_SEX,TRANSIT_DISTRICT,Latitude,Longitude,Lat_Lon,PATROL_BORO,STATION_NAME,VIC_AGE_GROUP,VIC_RACE,VIC_SEX
CMPLNT_NUM,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
325341655,02/11/2015,15:00:00,NaN,NaN,73.0,02/11/2015,359,OFFENSES AGAINST PUBLIC ADMINI,749.0,VIOLATION OF ORDER OF PROTECTI,COMPLETED,MISDEMEANOR,BROOKLYN,INSIDE,RESIDENCE - PUBLIC HOUSING,N.Y. HOUSING POLICE,2.0,NaN,TILDEN,405,1009656.0,181287.0,25-44,BLACK,M,NaN,40.664239,-73.908425,"(40.664239422, -73.908425011)",PATROL BORO BKLYN NORTH,NaN,<18,BLACK,M
393816841,03/17/2012,10:30:00,03/17/2012,11:00:00,69.0,03/17/2012,344,ASSAULT 3 & RELATED OFFENSES,114.0,OBSTR BREATH/CIRCUL,COMPLETED,MISDEMEANOR,BROOKLYN,INSIDE,RESIDENCE-HOUSE,N.Y. POLICE DEPT,0.0,NaN,NaN,NaN,1014035.0,174133.0,NaN,NaN,NaN,NaN,40.644590,-73.892672,"(40.644589618, -73.892672426)",PATROL BORO BKLYN SOUTH,NaN,45-64,BLACK,F
802896158,10/27/2016,13:48:00,11/03/2016,13:49:00,71.0,11/03/2016,578,HARRASSMENT 2,638.0,"HARASSMENT,SUBD 3,4,5",COMPLETED,VIOLATION,BROOKLYN,INSIDE,PUBLIC SCHOOL,N.Y. POLICE DEPT,0.0,NaN,NaN,NaN,1000222.0,179282.0,<18,BLACK,M,NaN,40.658758,-73.942435,"(40.658758183, -73.942434788)",PATROL BORO BKLYN SOUTH,NaN,18-24,BLACK,M
633812343,11/27/2014,19:00:00,11/27/2014,22:30:00,112.0,11/28/2014,104,RAPE,157.0,RAPE 1,COMPLETED,FELONY,QUEENS,INSIDE,RESIDENCE - APT. HOUSE,N.Y. POLICE DEPT,0.0,NaN,NaN,NaN,1025420.0,202485.0,25-44,WHITE HISPANIC,M,NaN,40.722364,-73.851474,"(40.722363687, -73.851473894)",PATROL BORO QUEENS NORTH,NaN,25-44,WHITE,F
300349533,12/11/2013,13:30:00,12/11/2013,14:15:00,24.0,12/12/2013,109,GRAND LARCENY,438.0,"LARCENY,GRAND FROM BUILDING (NON-RESIDENCE) UN...",COMPLETED,FELONY,MANHATTAN,INSIDE,DOCTOR/DENTIST OFFICE,N.Y. POLICE DEPT,0.0,NaN,NaN,NaN,992848.0,228356.0,NaN,NaN,NaN,NaN,40.793465,-73.968950,"(40.793464597, -73.968949638)",PATROL BORO MAN NORTH,NaN,45-64,WHITE,F


In [ ]:
crime_df_yrs.drop(columns=['Latitude', 'Longitude', 'X_COORD_CD', 'Y_COORD_CD', 'JURIS_DESC', 'JURISDICTION_CODE', 'HADEVELOPT', 'HOUSING_PSA'], inplace=True)

# Separate crimes by years and saves each dataframe keyed in a dictionary

In [ ]:
def seperate_by_yr(df):
  year_dict = {}

  # Grab only the year and make it an int type
  out_df = df.copy(deep=True)
  out_df['Year'] = out_df['CMPLNT_FR_DT'].str[6:].fillna(0)
  out_df['Year'] = out_df['Year'].astype('int16')

  # Get only 2006-2019
  latest_yr, earliest_yr = 2006, 2020
  out_df = out_df[out_df['Year'] > latest_yr] 
  for yr in range(latest_yr, earliest_yr):
    select_yr = out_df[out_df['Year'] == yr]
    year_dict[yr] = select_yr
  
  return year_dict

In [ ]:
sep_year_crime = seperate_by_yr(crime_df_yrs)

In [ ]:
for yr in range(2007, 2020):
  sep_year_crime[yr].to_csv('years/{}.csv'.format(yr))

In [ ]:
!git status

fatal: bad object HEAD
